In [1]:
import tensorflow_datasets as tfds
import tensorflow as tf

/home/polivares/miniforge3/envs/TensorflowGPUv2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2023-10-29 19:17:09.001105: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2023-10-29 19:17:09.036653: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-10-29 19:17:09.676017: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT W

In [2]:
dataset, info = tfds.load("tf_flowers", as_supervised=True, with_info=True)

2023-10-29 19:17:10.438931: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:996] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2023-10-29 19:17:10.483475: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:996] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2023-10-29 19:17:10.483604: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:996] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysf

In [3]:
test_set, valid_set, train_set = tfds.load("tf_flowers",split=["train[0%:10%]", "train[10%:25%]", "train[25%:]"], as_supervised=True )

In [4]:
# import tensorflow as tf

In [5]:
# Previo a la utilización de nuestro dataset en la red neuronal, la red que utilizaremos provee de
# funcionalidades de pre-procesamiento ya incluídas. Por lo tanto, utilizaremos estas funcionalidades
# y las dejaremos dentro de una función
def preprocess(image, label):
    # Cambiaremos las dimensiones de la imagen de entrada
    resized_image = tf.image.resize(image, [224, 224]) # Guardamos la imagen con nuevas dimensiones 224x224
    # Luego pasamos la imagen modificada en tamaño al preprocesamiento de nuestra red. La red
    # que utilizaremos de ejemplo tiene por nombre Xception
    final_image = tf.keras.applications.xception.preprocess_input(resized_image)
    # Finalmente se retorna una imagen pre procesada (según lo indique el preprocess de xception)
    # y su etiqueta
    return final_image, label

In [6]:
# En este apartado hacemos los batch de datos directamente desde el dataset cargado por tensorflow
batch_size = 32

# Mezclamos el dataset 
train_set = train_set.shuffle(1000)
# Tanto para training, test y validación, aplicamos la función de preprocesamiento (preprocess)
# y luego generamos los batch de datos
train_set = train_set.map(preprocess).batch(batch_size).prefetch(1)
test_set = test_set.map(preprocess).batch(batch_size).prefetch(1)
valid_set = valid_set.map(preprocess).batch(batch_size).prefetch(1)

In [7]:
# Acá empezamos con transfer learning. Lo que haremos será cargar una arquitectura de red neuronal desde
# tensorflow ya entrenada. Eso quiere decir que, no solo estamos cargando la arquitectura, con sus neuronas
# y conexiones, sino que también estamos cargando los PESOS DE ENTRENAMIENTO.

# Weights indica si utilizaremos pesos pre entrenados con el dataset imagenet o no
# include_top es el parámetro que indica explícitamente si quieres o no la capa de salida original
# de esta red.
# En base_model tenemos cargado nuestro modelo Xception sin la capa de salida. Nosotros podemos
# poner NUESTRA PROPIA CAPA(S) DE SALIDA
base_model = tf.keras.applications.xception.Xception(weights="imagenet", include_top=False)
# Acá agregamos nuestras capas adicionales
avg = tf.keras.layers.GlobalAveragePooling2D()(base_model.output)
output = tf.keras.layers.Dense(5, activation="softmax")(avg)

# Podemos conectar el input de nuestro modelo base con el output recién generado a través de 
# un modelo de Keras
model = tf.keras.Model(inputs=base_model.input, outputs=output)
# En este punto tenemos un nuevo modelo que utiliza como base la arquitectura Xception

In [8]:
# También nosotros conversamos que es posible decidir si queremos reentrenar aquellas capas ya entrenadas

# Acá vamos capa por capa modificando el parámetro "trainable" que permite (o no) reentrenar dicha capa
for layer in base_model.layers:
    layer.trainable = False # Esto impide que las capas se re entrenen

# Con este for, sobre todas las capas de nuestro modelo, estamos impidiendo que se reentrene
# alguna de sus capas ya entrenadas

In [9]:
# Proceso de compilación  (tal cual vimos en las clases anteriores)
model.compile(loss="sparse_categorical_crossentropy", optimizer="sgd", 
              metrics=["accuracy"])

In [10]:
history = model.fit(train_set, epochs=10, validation_data=valid_set)

Epoch 1/10


2023-10-29 19:17:12.238703: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_3' with dtype int64 and shape [2]
	 [[{{node Placeholder/_3}}]]
2023-10-29 19:17:12.239088: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_2' with dtype string and shape [2]
	 [[{{node Placeholder/_2}}]]
2023-10-29 19:17:15.287641: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:417] Loaded runtime CuDNN library: 8.3.0 but source was compiled with: 8.6.0.  CuDNN library needs to have matching major version and equal or higher minor version. If using a binary install, upgrade your CuDNN library.  If building

UnimplementedError: Graph execution error:

Detected at node 'model/block1_conv1/Conv2D' defined at (most recent call last):
    File "<frozen runpy>", line 198, in _run_module_as_main
    File "<frozen runpy>", line 88, in _run_code
    File "/home/polivares/miniforge3/envs/TensorflowGPUv2/lib/python3.11/site-packages/ipykernel_launcher.py", line 17, in <module>
      app.launch_new_instance()
    File "/home/polivares/miniforge3/envs/TensorflowGPUv2/lib/python3.11/site-packages/traitlets/config/application.py", line 1053, in launch_instance
      app.start()
    File "/home/polivares/miniforge3/envs/TensorflowGPUv2/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 737, in start
      self.io_loop.start()
    File "/home/polivares/miniforge3/envs/TensorflowGPUv2/lib/python3.11/site-packages/tornado/platform/asyncio.py", line 195, in start
      self.asyncio_loop.run_forever()
    File "/home/polivares/miniforge3/envs/TensorflowGPUv2/lib/python3.11/asyncio/base_events.py", lin

In [ ]:
model.predict(test_set)

In [ ]:
import numpy as np
for image, label in test_set:
    print("Etiqueta real",label)
    print("Predicción", model.predict(image))
